In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab06.ipynb")

# Lab 06 — Missing Values and Imputation

In this lab, you will identify missingness mechanisms, use K–S and permutation tests to investigate missingness, and apply mean and group-based imputation to height data.

Complete all questions and submit your notebook to **Gradescope by the deadline listed in Canvas**. Before submitting, restart the kernel and run all cells to verify that your notebook executes successfully and passes the public tests.

**We recommend submitting early rather than waiting until the last moment, so you have time to resolve technical issues and seek support from course staff.**

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />


## Part 1: Missingness Mechanisms

First, let's recap the different mechanisms of missingness we studied in lecture.

#### Missing by Design (MD)
- The missing field is deliberately missing. The missing field is deliberately set to null or not collected (hence, "missing by design").
- The missingness can be exactly predicted when a column will be null, with only knowledge of the other columns using a function of the rows of the dataset.

#### Missing Completely at Random (MCAR)
- The missingness of missing value isn't related to the actual, unreported value itself, nor the values in any other fields. The missingness is not systematic.
- The missingness is unconditionally uniform across rows. MCAR doesn't bias the observed data.
- There is no relationship between the missing data and the any of the other data, observed or missing.

#### Missing at Random (MAR)
- The missingness of the missing value has nothing to do with the value itself, but may be related to another field.
- The missingness is uniform across rows, perhaps conditional on another column. MAR biases the observed data, but is fixable.
- There is a systematic relationship between the missing values and the observed data (but not the missing values themselves).
- Difference between MD and MAR: If you can *exactly/always* determine missingness using the other columns, the missingness is MD. If there is just some sort of systematic relationship between the missing columns/values and other columns/values that may help us predict missingness, the missingness is MAR.

#### Not Missing At Random (NMAR)
- The missingness of the missing value is related to the actual, unreported value.
- NMAR biases the observed data in unobservable ways.
- There is relationship between the propensity of a value to be missing and its value.

### Dataset: Height Data

The file `data/missing_heights.csv` contains adult children’s heights and their fathers’ heights, measured in inches.

- **`father`**: Father’s height.
- **`child`**: Child’s height, with no missing values.
- **`child_X`**: Child’s height with some values intentionally removed; `X` indicates the percentage of values retained.

For this lab, use **`child_50`**, where approximately half of the child heights are missing.

In [ ]:
heights = pd.read_csv(Path("data") / "missing_heights.csv")
heights.head()

<br>

--- 

### Question 1 — 10 points

You run a small e-commerce website and send surveys out to customers after they purchase an item from your store. The survey asks whether the customer is satisfied with their purchase ("Yes" or "No"). Below, you are presented with possible datasets, each of which contains a column `'satisfied'` as described above, as well as a `'customer_id'` number corresponding to the customer and an `'item'` column describing the item that the customer purchased. **The column `'satisfied'` is missing data.**

For each of the following datasets, label the column `'satisfied'` as being `'MD'`, `'MCAR'`, `'MAR'`, or `'NMAR'`.

1. The dataset consists only of the columns `'customer_id'` and `'satisfied'`.
2. The dataset contains the `'customer_id'` of every customer with an account, even if they didn't make a purchase. Also, in this case, you notice everyone who was sent a survey filled it out.
3. The dataset contains a column specifying if the user later returned the item.
4. The dataset contains a column with the serial number for the item purchased.
5. The dataset contains a column with the price of the item purchased.

Complete the implementation of the function `after_purchase`, which records your answers and returns a list of length 5, containing the values `'MD'`, `'MCAR'`, `'MAR'`, or `'NMAR'`. For some questions there may be multiple good answers, but there is generally one answer that is "best". 

***Disclaimer***: It is possible to just look at the some of the correct answers by running `grader.check`. This is not a good idea – you should really think about all of the questions here, since similar questions will be on the exams.

In [ ]:
def after_purchase(): 
    # Return a list with the type of missingness for the 5 datasets.
    #  Types are 'MD', 'MCAR', 'MAR', 'NMAR'
    
    return None
q1_ans = after_purchase()

In [ ]:
grader.check("q1")

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Part 2: Comparing Distributions with K–S Tests

The two-sample Kolmogorov–Smirnov (K–S) test compares numerical distributions. Its statistic is the largest absolute difference between their empirical cumulative distribution functions.

- **Null hypothesis:** The two samples come from the same distribution.
- **Small p-value:** Evidence that the distributions differ.
- **Large p-value:** Insufficient evidence of a difference; it does not prove the distributions are identical.

Use `stats.ks_2samp(..., method="auto")` for these questions. First compare father and child heights, then compare father heights between rows where `child_50` is missing and present. The second comparison investigates whether missingness is associated with father height; it cannot establish the full missingness mechanism.


### Question 2 — 15 points

Run a **two-sample K–S test** to compare the complete `father` and `child` height distributions. Store the test statistic in `height_stat` and the p-value in `height_pvalue`. Use `method="auto"`.

*Hint:* The test result has `.statistic` and `.pvalue` attributes.

In [ ]:
height_result = ...
height_stat = ...
height_pvalue = ...

In [ ]:
grader.check("q2")

### Question 3 — 15 points

Complete `missingness_ks(df)` to investigate whether `father` heights differ depending on whether `child_50` is missing.

Inside the function:

1. Create `mask` to identify rows where `child_50` is missing.
2. Compare `father` heights in those rows with `father` heights in rows where `child_50` is present. Use `stats.ks_2samp` with `method="auto"` and store the output in `result`.
3. Return the statistic and p-value from `result` as a tuple of floats.

Assume both groups contain observations.

*Hint:* Use `isna()` to create the mask and `~mask` to select the other group.

In [ ]:

def missingness_ks(df):
    mask = ... 
    return = ...
    return ...

In [ ]:
grader.check("q3")

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Part 3: Investigating Missingness with Permutation Tests

Compare father heights between rows where `child_50` is missing and present, using the **absolute difference between the group means**.

Under the null hypothesis of identical group distributions, the group labels are exchangeable:

1. Pool the father heights and shuffle them.
2. Split the shuffled values while preserving the original group sizes.
3. Calculate the absolute difference between the shuffled group means.
4. Repeat and estimate the p-value as the proportion of simulated statistics at least as large as the observed statistic.

A small p-value provides evidence of an association between child-height missingness and father height. A large p-value does not prove MCAR.


In [ ]:
missing_mask = heights["child_50"].isna()
father_missing = heights.loc[missing_mask, "father"].to_numpy()
father_present = heights.loc[~missing_mask, "father"].to_numpy()

### Question 4 — 15 points

Complete **one permutation trial** using the supplied `shuffled` father heights.

- Store the two shuffled groups in `shuffled_missing` and `shuffled_present`, preserving the sizes of `father_missing` and `father_present`, respectively.
- Store the **absolute difference between their means** as a float in `permuted_stat`.

Do not change the setup code.

*Hint:* The first group has len(father_missing) values.

In [ ]:
# Do not change the setup.
rng = np.random.default_rng(42)
pooled = np.concatenate([father_missing, father_present])
shuffled = rng.permutation(pooled)
n_missing = len(father_missing)

shuffled_missing = ...
shuffled_present = ...
permuted_stat = ...

In [ ]:
grader.check("q4")

### Question 5 — 15 points

The provided code generates 1,000 permutation statistics to compare father heights between rows where `child_50` is missing and present.

Complete the missing code to determine:

- `observed_stat`: The observed absolute difference between the two group means.
- `permutation_p`: The permutation-test p-value.
- `reject_null`: Set `reject_null` to `True` if this p-value is below 0.05, otherwise `False`.

Run the provided simulation without editing it.

*Hint:* A Boolean array can be averaged.

In [ ]:
# Provided simulation: run without editing.
rng = np.random.default_rng(2201)
pooled = np.concatenate([father_missing, father_present])
n_missing = len(father_missing)
permutation_stats = np.empty(1000)
for i in range(1000):
    shuffled_values = rng.permutation(pooled)
    permutation_stats[i] = abs(shuffled_values[:n_missing].mean()
                               - shuffled_values[n_missing:].mean())

observed_stat = ...
permutation_p = ...
reject_null = ...

In [ ]:
grader.check("q5")

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Part 4: Imputation

Imputation fills missing values using information from the observed data. Keep all originally observed values unchanged.

#### Overall Mean Imputation
- Fill missing child heights with the mean of the observed child heights.
- This preserves the observed mean but reduces variability, which can affect later analysis.

#### Group-Based Mean Imputation
- Divide the data into the supplied father-height groups.
- Fill missing child heights with the observed child-height mean within the same group.
- This accounts for differences between groups, but does not fully restore the variability or uncertainty in the missing values.


### Question 6 — 15 points

Complete the code to fill missing heights in `child_50` heights using the mean of its nonmissing values.

- Store the filled Series in `mean_filled`, keeping all originally observed values unchanged.
- Calculate the sample standard deviation of `mean_filled` and store it in `filled_std`.

*Hint:* Use `fillna()` and `std()`.

In [ ]:
mean_filled = ...
filled_std = ...

In [ ]:
grader.check("q6")

### Question 7 — 15 points

Complete `group_impute(df)` to fill missing `child_50` heights with the observed mean child height within the same father-height group. The starter code defines two groups: `father < 70` and `father >= 70`.

- `group_means`: The observed mean child height for each row’s group.
- `filled`: The completed Series, with observed values unchanged.

Return `filled` with the original index. Assume each group contains at least one observed child height.

*Hint:* Use `groupby`, `transform`, and `fillna`.

In [ ]:

def group_impute(df):
    groups = df["father"] >= 70
    group_means = ...
    filled = ...
    return filled

In [ ]:
grader.check("q7")

## Congratulations! You have completed Lab 06!

Before submitting:

- Select **Kernel → Restart & Run All**.
- Confirm that all cells run successfully and all public tests pass.
- Save your notebook and submit it to **Gradescope**.

If tests fail after restarting, check for missing variables, deleted cells, or code that depends on earlier changes to your DataFrame.

You may resubmit as many times as needed before the deadline. Passing public tests does not guarantee passing all hidden tests.

**Always restart the kernel and run all cells before submitting. Regrade requests based solely on tests passing locally will not be accepted.**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)